In [1]:
import pandas as pd
import json
import plotly.express as px

In [2]:
#read in csv
cols =['date-event-began', 'nerc-region', 'area-affected',
       'loss-(megawatts)', 'number-of-customers-affected', 'year', 'cause','date_event_restored']
power_df = pd.read_csv('power_df.csv',dtype={'number-of-customers-affected':float},usecols=cols)
power_df.head()

,date-event-began,nerc-region,area-affected,loss-(megawatts),number-of-customers-affected,year,cause,date_event_restored
0,06:00:00,SPP,Pennsylvania,500.0,1881134.0,2002,Weather Events,2002-02-07 12:00:00
1,15:00:00,SPP,Alabama,550.0,270000.0,2002,Weather Events,NaN
2,16:00:00,SPP,Missouri,210.0,95000.0,2002,Weather Events,2002-02-10 21:00:00
3,10:48:00,WSCC,Kentucky,300.0,255000.0,2002,Transmission Issues,2002-02-27 11:35:00
4,00:00:00,ECAR,Ohio,190.0,190000.0,2002,Weather Events,2002-03-11 12:00:00


In [3]:
power_df.isna().sum()

date-event-began                  22
nerc-region                        0
area-affected                     49
loss-(megawatts)                1350
number-of-customers-affected     607
year                               0
cause                             44
date_event_restored               34
dtype: int64

In [7]:
fig = px.line(power_df.groupby('year')['area-affected'].nunique(),
              title='How many unique states recorded power outage events each year?')
fig.update_layout(showlegend = False,
                  yaxis_title = 'Number of states')
fig.show()

In [14]:
#get number of events recorded by year per cause
df_grouped = power_df.groupby(['year','cause']).cause.count().reset_index(name='count')
# Plot
fig = px.line(
    df_grouped,
    x='year',
    y='count',
    color='cause',
    markers=True,
    title='Power Outage Trends by Cause Over the Years'
)

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Number of Outages',
    legend_title='Cause',
    template='plotly_white'
)
fig.show()


In [ ]:
#get number of evens recorded by each state per cause
power_df.groupby(['area-affected','cause']).cause.count()

area-affected  cause                                 
Alabama        Public Alerts                              1
               Security and Vandalism                     8
               Suspicious Activities and Cyber Events     5
               System & Fuel Issues                       2
               Transmission Issues                        2
                                                         ..
Wisconsin      Weather Events                            27
Wyoming        Security and Vandalism                     7
               Suspicious Activities and Cyber Events     1
               Transmission Issues                        6
               Weather Events                             6
Name: cause, Length: 209, dtype: int64

In [4]:
# read in the dictionary with state abbreviations
with open("state_to_abbr.json", "r") as f:
    STATE_TO_ABBR = json.load(f)
# Display the first key-value pairs
print(list(STATE_TO_ABBR.items())[:5] )

[('New York', 'NY'), ('California', 'CA'), ('Illinois', 'IL'), ('Florida', 'FL'), ('Texas', 'TX')]


In [5]:
# create a state column with two letter abrreviations
power_df['state'] = power_df['area-affected'].map(STATE_TO_ABBR)
power_df['state'].value_counts().head()

state
KY    434
MO    342
IL    298
VA    274
PA    219
Name: count, dtype: int64

In [10]:
year = power_df['year'].value_counts().sort_index()
fig = px.bar(year, title='Are outages becoming more or less frequent each year?')
fig.update_layout(showlegend = False,
                  yaxis_title = 'event counts per year')
fig.show()

In [18]:
fig = px.line(power_df.groupby(['year'])['loss-(megawatts)'].max(),
              title='What was the maximum power loss in each year?')
fig.update_layout(showlegend = False,
                  yaxis_title = 'power loss in MW')
fig.show()

In [ ]:
fig = px.line(power_df.groupby(['year'])['number-of-customers-affected'].mean(),
              title='Has the  average number of people affected increased or decreased over time?')
fig.update_layout(showlegend = False,
                  yaxis_title = 'Number of people affeced')
fig.show()

In [ ]:
# plot showing which cause of alert caused the most damage interms of power loss and people affected per state
df = power_df[power_df['cause'] == 'Weather Events']
avg = df.groupby('area-affected', as_index=False)['number-of-customers-affected'].mean()
avg = avg.sort_values(by='number-of-customers-affected', ascending=True).tail(10).dropna()
fig = px.bar(
            avg,
            x='number-of-customers-affected',
            y='area-affected',
            orientation='h',
            labels={'number-of-customers-affected': 'Average', 'area-affected': 'State'},
            title=f"Top States by Avg {'number-of-customers-affected'.capitalize()} - {'Weather Events'}"
        )
fig.show()

In [25]:
df = power_df.groupby(['year', 'cause'], as_index=False)['loss-(megawatts)'].sum()
fig = px.line(
            df,
            x='year',
            y='loss-(megawatts)',
            color='cause',
            markers=True,
            title=f"Yearly Total {'loss-(megawatts)'.capitalize()} by Cause"
        )
fig.show()

In [31]:
#create a state count column using frequency encoding
power_df['state_count'] = power_df['area-affected'].map(power_df['area-affected'].value_counts())
#create a chooropleth to visualize the number of usa states that received power challenges in our data
fig = px.choropleth(power_df,
                     locations = 'state',
                     locationmode="USA-states",
                     hover_name= 'area-affected',
                     scope='usa',
                     color='state_count',
                     color_continuous_scale=px.colors.sequential.Greens,
                     title='map showing number of usa states that faced power challenges')
fig.show()

In [32]:
# create a visualization to help us understand nerc region
fig = px.choropleth(power_df,
                     locations = 'state',
                     locationmode="USA-states",
                     hover_name= 'nerc-region',
                     scope='usa',
                     color='nerc-region',
                     color_discrete_sequence=px.colors.sequential.Oranges,
                     title='map showing nerc regions that faced power challenges')
fig.show()

In [33]:
# filter a df
weather = power_df.query("cause =='Weather Events'").copy()
weather['count'] = weather['state'].map(weather['state'].value_counts())
fig = px.choropleth(weather,
                     locations = 'state',
                     locationmode="USA-states",
                     hover_name= 'area-affected',
                     scope='usa',
                     color='count',
                     color_continuous_scale=px.colors.sequential.Greens,
                     title='map showing number of usa states that faced power challenges due to weather')
fig.show()

In [34]:
def plot_cause(selected_cause):
    """filters the power_df, creates a count column and returns a fig"""
    #filter the df basing on the cause column and create a copy
    df = power_df.query("cause == @selected_cause").copy()
    #create a count column
    df['count'] = df['state'].map(df['state'].value_counts())
    #create a fig
    fig = px.choropleth(df,
                        locations = 'state',
                        locationmode="USA-states",
                        hover_name= 'area-affected',
                        scope='usa',
                        color='count',
                        color_continuous_scale=px.colors.sequential.Greens,
                        title=f"map showing number of usa states that faced power challenges due to {selected_cause}")
    #fig.show()
    return fig

In [35]:
plot_cause('Transmission Issues')

In [36]:
#creating a visualization for average loss

# Convert the 'loss-(megawatts)' column to numeric, coercing errors to NaN
power_df['loss-(megawatts)'] = pd.to_numeric(power_df['loss-(megawatts)'], errors='coerce')

# Group by 'area-affected' and calculate the mean
avg_loss = power_df.groupby('area-affected')['loss-(megawatts)'].mean()

# Create the 'avg_loss' column in power_df by mapping the avg_loss values
power_df['avg_loss'] = power_df['area-affected'].map(avg_loss)

# Create a choropleth map
fig = px.choropleth(power_df,
                    locations='state',
                    locationmode="USA-states",
                    hover_name='area-affected',
                    scope='usa',
                    color='avg_loss',
                    color_continuous_scale=px.colors.sequential.Greens,
                    title="Map Showing Average Power Loss Due to Power Challenges")

fig.show()

In [37]:
#creating a visualization for number of affected people

# Convert the 'loss-(megawatts)' column to numeric, coercing errors to NaN
power_df['number-of-customers-affected'] = pd.to_numeric(power_df['number-of-customers-affected'], errors='coerce')

# Group by 'area-affected' and calculate the mean
avg_people_affected = power_df.groupby('area-affected')['number-of-customers-affected'].mean()

# Create the 'avg_loss' column in power_df by mapping the avg_loss values
power_df['avg_people_affected'] = power_df['area-affected'].map(avg_people_affected)

# Create a choropleth map
fig = px.choropleth(power_df,
                    locations='state',
                    locationmode="USA-states",
                    hover_name='area-affected',
                    scope='usa',
                    color='avg_people_affected',
                    color_continuous_scale=px.colors.sequential.Greens,
                    title="Map Showing Average People affected Due to Power Challenges")

fig.show()

In [38]:
fig = px.bar(data_frame=power_df['nerc-region'].value_counts(normalize=True).head(10).sort_values(ascending=True),
             orientation= 'h',
             width = 500,
             height= 400)
fig.update_layout(xaxis_title = 'normalized value counts',
                  title ='Bar plot showing the top ten nerc region',
                  showlegend=False,  
                  xaxis=dict(range=[0, 1]))
fig.show()

In [12]:
fig = px.bar(data_frame=power_df['area-affected'].value_counts().tail(10).sort_values(ascending=True),
             orientation= 'h',
             width = 500,
             height= 400)
fig.update_layout(xaxis_title = 'counts',
                  title ='Plot showing top areas affected by power challenges',
                  showlegend=False) 
                  
fig.show()

In [40]:
fig = px.bar(data_frame=power_df['cause'].value_counts().sort_values(ascending=True),
             orientation= 'h',
             width = 500,
             height= 400)
fig.update_layout(xaxis_title = 'counts',
                  title ='Plot showing top areas affected by power challenges',
                  showlegend=False) 
                  
fig.show()

In [41]:
# Convert the 'loss-(megawatts)' column to numeric, coercing errors to NaN
power_df['loss-(megawatts)'] = pd.to_numeric(power_df['loss-(megawatts)'], errors='coerce')

# Group by 'area-affected' and calculate the mean
avg_loss = power_df.groupby('area-affected')['loss-(megawatts)'].mean()
#create a bar plot
fig = px.bar(data_frame= avg_loss.sort_values(ascending=False).head().sort_values(ascending=True),
             orientation='h',
             width = 500,
             height= 400)
fig.update_layout(xaxis_title = 'average loss in megawatts',
                  title ='Plot showing states with highest average power loss',
                  showlegend=False) 
fig.show()

In [42]:
def plot_filtered_bar(col_name,selected_cause):
    """filters power_df, calculates value counts and returns a fig"""
    #filter the df basing on the cause column and create a copy
    df = power_df.query("cause == @selected_cause").copy()
    # Convert the selected column to numeric, coercing errors to NaN
    df[col_name] = pd.to_numeric(df[col_name], errors='coerce')

    # Group by 'area-affected' and calculate the mean
    grouped_data = df.groupby('area-affected')[col_name].mean()
    #create a bar plot
    fig = px.bar(data_frame= grouped_data.sort_values(ascending=False).head().sort_values(ascending=True),
                orientation='h',)
                #width = 500,
                #height= 400)
    fig.update_layout(xaxis_title = f'average {col_name}',
                    title =f'Plot showing states with highest average {col_name} due to {selected_cause}',
                    showlegend=False) 
    return fig

In [43]:
plot_filtered_bar('loss-(megawatts)','Transmission Issues')